# 🏗️ Step 1: Setup Catalog & Generate Synthetic Security Data

This notebook:
1. Creates the schema (in the pre-existing catalog)
2. Generates synthetic security telemetry mimicking an MDR provider's data
3. Creates reference tables (threat intel, asset inventory, customers)
4. Creates historical triage decisions for ML training

**Run time**: ~3-5 minutes

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
# Catalog already exists — using rajarora_febar
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {LANDING_VOLUME} "
          "COMMENT 'Landing zone for raw security telemetry files (ingested by Auto Loader)'")

# Full regeneration = fresh environment: clear landing files, the Auto Loader checkpoint and bronze,
# so 02a re-ingests exactly this run's files (no duplicates from an earlier run)
dbutils.fs.rm(LANDING_PATH, True)
dbutils.fs.rm(CHECKPOINT_PATH, True)
spark.sql(f"DROP TABLE IF EXISTS {TABLES['raw_security_events']}")
print(f"✅ Schema '{FULL_SCHEMA}' and volume '{LANDING_VOLUME}' ready (landing zone reset)")

✅ Schema 'rajarora_febar.soc_alert_triage' and volume 'rajarora_febar.soc_alert_triage.landing' ready (landing zone reset)


In [0]:
import random
from pyspark.sql import functions as F
from pyspark.sql.types import *

random.seed(42)

industries = ["Healthcare", "Financial Services", "Manufacturing", "Retail", "Technology", 
              "Energy", "Government", "Education", "Legal", "Logistics"]
tiers = ["Enterprise", "Mid-Market", "SMB"]
regions = ["US-East", "US-West", "US-Central", "EMEA", "APAC", "LATAM"]

# 10 x 10 = 100 unique company names, shuffled — one per customer (names must be unique for reporting)
company_names = [f"{a} {b}" for a in ['Acme','Global','Pacific','Atlantic','Summit','Pinnacle','Vertex','Apex','Horizon','Nexus']
                 for b in ['Corp','Inc','LLC','Group','Holdings','Systems','Solutions','Partners','Industries','Technologies']]
random.shuffle(company_names)

customers_data = []
for i in range(1, 101):
    customers_data.append({
        "customer_id": f"CUST-{i:04d}",
        "customer_name": company_names[i - 1],
        "industry": random.choice(industries),
        "tier": random.choice(tiers),
        "region": random.choice(regions),
        "num_endpoints": random.randint(50, 10000),
        "num_users": random.randint(20, 5000),
        "contract_start_date": f"202{random.randint(3,5)}-{random.randint(1,12):02d}-01",
        "is_active": True
    })

customers_df = spark.createDataFrame(customers_data)
customers_df.write.mode("overwrite").saveAsTable(TABLES["customers"])
print(f"✅ Created {customers_df.count()} customers")
customers_df.display()

✅ Created 100 customers


contract_start_date,customer_id,customer_name,industry,is_active,num_endpoints,num_users,region,tier
2025-08-01,CUST-0001,Summit LLC,Retail,true,5205,1761,LATAM,SMB
2023-04-01,CUST-0002,Summit Inc,Government,true,2390,2189,EMEA,SMB
2025-07-01,CUST-0003,Nexus Inc,Legal,true,9627,3529,US-Central,SMB
2023-01-01,CUST-0004,Acme Technologies,Energy,true,8398,4062,US-West,Enterprise
2025-02-01,CUST-0005,Vertex Systems,Financial Services,true,2671,3478,LATAM,Enterprise
2024-09-01,CUST-0006,Pinnacle Corp,Government,true,7718,4354,APAC,Mid-Market
2024-11-01,CUST-0007,Acme Inc,Healthcare,true,1926,4418,LATAM,SMB
2024-01-01,CUST-0008,Apex Corp,Energy,true,7173,1315,US-Central,Enterprise
2025-05-01,CUST-0009,Global Systems,Technology,true,8367,891,US-West,SMB
2023-09-01,CUST-0010,Apex Industries,Legal,true,2554,3083,US-West,SMB


In [0]:
import ipaddress

ioc_types = ["ip_address", "domain", "file_hash", "url", "email"]
threat_categories = ["malware", "phishing", "c2_server", "botnet", "ransomware", "apt", "cryptominer"]
confidence_levels = ["high", "medium", "low"]
sources = ["AlienVault OTX", "VirusTotal", "AbuseIPDB", "PhishTank", "MalwareBazaar", "MISP", "ThreatFox"]

iocs_data = []
for i in range(10000):
    ioc_type = random.choice(ioc_types)
    if ioc_type == "ip_address":
        value = str(ipaddress.IPv4Address(random.randint(1, 2**32 - 1)))
    elif ioc_type == "domain":
        value = f"{random.choice(['evil','bad','malware','phish','hack','dark','shadow','storm'])}{random.randint(1,9999)}.{random.choice(['com','net','org','xyz','top','ru','cn'])}"
    elif ioc_type == "file_hash":
        value = ''.join(random.choices('0123456789abcdef', k=64))
    elif ioc_type == "url":
        value = f"http://{random.choice(['evil','bad','malware'])}{random.randint(1,999)}.com/{random.choice(['payload','download','login','update'])}"
    else:
        value = f"{random.choice(['admin','support','billing'])}@{random.choice(['evil','phish','scam'])}{random.randint(1,99)}.com"
    
    iocs_data.append({
        "ioc_id": f"IOC-{i+1:06d}",
        "ioc_type": ioc_type,
        "ioc_value": value,
        "threat_category": random.choice(threat_categories),
        "confidence": random.choice(confidence_levels),
        "source": random.choice(sources),
        "first_seen": f"202{random.randint(3,6)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        "is_active": random.random() > 0.1
    })

iocs_df = spark.createDataFrame(iocs_data)
iocs_df.write.mode("overwrite").saveAsTable(TABLES["threat_intel_iocs"])
print(f"✅ Created {iocs_df.count()} threat intelligence IOCs")
iocs_df.display()

✅ Created 10000 threat intelligence IOCs


confidence,first_seen,ioc_id,ioc_type,ioc_value,is_active,source,threat_category
high,2025-02-08,IOC-000001,file_hash,5eb40f1b7cba7c13b49786b5b441318c237bf841332084f8b2d7d97720f7d61a,true,VirusTotal,malware
low,2026-08-10,IOC-000002,url,http://malware611.com/download,true,PhishTank,cryptominer
low,2024-11-07,IOC-000003,url,http://bad583.com/payload,true,AlienVault OTX,ransomware
high,2023-03-01,IOC-000004,ip_address,40.52.228.193,true,MalwareBazaar,phishing
medium,2025-12-28,IOC-000005,email,support@phish5.com,true,MISP,phishing
high,2023-09-08,IOC-000006,domain,hack9664.ru,true,PhishTank,cryptominer
low,2023-01-21,IOC-000007,file_hash,d124b941bbf87106541fd9ffa40c7ae29a7155ad5b4fd8755f68fa8635cac38e,true,PhishTank,botnet
low,2023-04-27,IOC-000008,domain,malware5092.top,true,PhishTank,malware
low,2025-07-27,IOC-000009,url,http://evil664.com/download,true,AbuseIPDB,botnet
high,2024-02-09,IOC-000010,url,http://evil468.com/download,true,MalwareBazaar,botnet


In [0]:
asset_types = ["server", "workstation", "laptop", "network_device", "cloud_instance", "container", "iot_device"]
os_types = ["Windows Server 2022", "Windows 11", "Ubuntu 22.04", "RHEL 9", "macOS Ventura", "Cisco IOS", "AWS Linux 2"]
criticality_levels = ["critical", "high", "medium", "low"]
departments = ["Engineering", "Finance", "HR", "Sales", "Marketing", "IT", "Executive", "Operations", "Legal", "R&D"]

assets_data = []
for i in range(5000):
    cust = random.choice(customers_data)
    assets_data.append({
        "asset_id": f"ASSET-{i+1:06d}",
        "customer_id": cust["customer_id"],
        "asset_type": random.choice(asset_types),
        "hostname": f"{random.choice(['srv','ws','lt','fw','vm','ct'])}-{random.randint(1,999):03d}.{cust['customer_name'].split()[0].lower()}.local",
        "ip_address": f"10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}",
        "os_type": random.choice(os_types),
        "criticality": random.choice(criticality_levels),
        "department": random.choice(departments),
        "is_managed": random.random() > 0.05,
        "last_seen": f"2026-09-{random.randint(25,30):02d}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:00Z"
    })

assets_df = spark.createDataFrame(assets_data)
assets_df.write.mode("overwrite").saveAsTable(TABLES["asset_inventory"])
print(f"✅ Created {assets_df.count()} assets")
assets_df.display()

✅ Created 5000 assets


asset_id,asset_type,criticality,customer_id,department,hostname,ip_address,is_managed,last_seen,os_type
ASSET-000001,laptop,low,CUST-0099,Finance,srv-264.global.local,10.46.35.125,true,2026-09-30T00:24:00Z,Windows Server 2022
ASSET-000002,network_device,low,CUST-0040,IT,lt-387.global.local,10.206.124.177,true,2026-09-29T21:53:00Z,AWS Linux 2
ASSET-000003,iot_device,low,CUST-0063,Marketing,ws-482.acme.local,10.19.148.218,true,2026-09-29T19:02:00Z,Cisco IOS
ASSET-000004,server,high,CUST-0042,Finance,vm-810.pacific.local,10.219.143.8,true,2026-09-30T11:09:00Z,macOS Ventura
ASSET-000005,container,high,CUST-0074,Operations,srv-843.nexus.local,10.113.64.125,true,2026-09-29T05:05:00Z,Windows Server 2022
ASSET-000006,cloud_instance,high,CUST-0055,Marketing,fw-555.global.local,10.30.189.156,true,2026-09-27T21:59:00Z,Windows Server 2022
ASSET-000007,server,critical,CUST-0048,HR,vm-899.vertex.local,10.253.142.102,true,2026-09-30T06:24:00Z,macOS Ventura
ASSET-000008,laptop,high,CUST-0072,HR,lt-518.nexus.local,10.168.73.238,true,2026-09-27T22:13:00Z,AWS Linux 2
ASSET-000009,container,low,CUST-0090,Operations,ct-222.vertex.local,10.163.174.247,true,2026-09-27T02:47:00Z,macOS Ventura
ASSET-000010,server,high,CUST-0077,Executive,ct-135.pacific.local,10.71.166.146,true,2026-09-28T05:14:00Z,macOS Ventura


### Event Types (mimicking MDR telemetry)
- **firewall**: Firewall allow/deny logs
- **edr**: Endpoint Detection & Response alerts
- **auth**: Authentication events (login success/failure)
- **email**: Email security events (phishing, spam, clean)
- **cloud_audit**: Cloud infrastructure audit logs (AWS CloudTrail style)

In [0]:
from datetime import datetime, timedelta
import json
import uuid

event_sources = {
    "firewall": {
        "actions": ["allow", "deny", "drop", "reset"],
        "protocols": ["TCP", "UDP", "ICMP", "HTTP", "HTTPS", "DNS", "SSH", "RDP"],
        "severities": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "weight": 0.35  # 35% of events
    },
    "edr": {
        "alert_types": ["malware_detected", "suspicious_process", "lateral_movement", 
                        "privilege_escalation", "data_exfiltration", "ransomware_behavior",
                        "fileless_attack", "credential_theft", "persistence_mechanism",
                        "process_injection"],
        "severities": [3, 4, 5, 6, 7, 8, 9, 10],
        "weight": 0.20
    },
    "auth": {
        "actions": ["login_success", "login_failure", "mfa_challenge", "mfa_failure",
                    "password_reset", "account_lockout", "privilege_change", "session_expired"],
        "auth_methods": ["password", "mfa_push", "mfa_totp", "sso", "certificate", "api_key"],
        "severities": [1, 2, 3, 4, 5, 6, 7, 8],
        "weight": 0.25
    },
    "email": {
        "verdicts": ["clean", "spam", "phishing", "malware_attachment", "suspicious_link",
                     "impersonation", "bec_attempt"],
        "severities": [1, 2, 3, 5, 7, 8, 9],
        "weight": 0.10
    },
    "cloud_audit": {
        "services": ["IAM", "S3", "EC2", "Lambda", "RDS", "CloudTrail", "VPC", "KMS"],
        "actions": ["CreateUser", "DeleteBucket", "StopInstance", "ModifySecurityGroup",
                    "AssumeRole", "PutBucketPolicy", "CreateAccessKey", "DisableLogging"],
        "severities": [1, 2, 3, 4, 5, 6, 7, 8, 9],
        "weight": 0.10
    }
}

# Pre-compute some malicious IPs from our IOC list for correlation
malicious_ips = [ioc["ioc_value"] for ioc in iocs_data if ioc["ioc_type"] == "ip_address"][:200]

NUM_EVENTS = 500000
base_time = datetime(2026, 9, 1, 0, 0, 0)

events_data = []
for i in range(NUM_EVENTS):
    # Pick event source based on weights
    r = random.random()
    cumulative = 0
    source_type = "firewall"
    for src, config in event_sources.items():
        cumulative += config["weight"]
        if r <= cumulative:
            source_type = src
            break
    
    cust = random.choice(customers_data)
    asset = random.choice([a for a in assets_data if a["customer_id"] == cust["customer_id"]] or [random.choice(assets_data)])
    
    # Generate timestamp spread over 30 days
    event_time = base_time + timedelta(
        days=random.randint(0, 29),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    
    severity = random.choice(event_sources[source_type]["severities"])
    
    # Inject ~5% true positive patterns
    is_true_positive = random.random() < 0.05
    if is_true_positive:
        severity = random.choice([7, 8, 9, 10])
    
    # Build source-specific payload
    if source_type == "firewall":
        src_ip = random.choice(malicious_ips) if is_true_positive else f"{random.randint(1,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
        payload = json.dumps({
            "action": "deny" if is_true_positive else random.choice(event_sources["firewall"]["actions"]),
            "protocol": random.choice(event_sources["firewall"]["protocols"]),
            "src_ip": src_ip,
            "dst_ip": asset["ip_address"],
            "src_port": random.randint(1024, 65535),
            "dst_port": random.choice([22, 80, 443, 445, 3389, 8080, 8443, 53]),
            "bytes_sent": random.randint(0, 100000),
            "bytes_received": random.randint(0, 500000)
        })
    elif source_type == "edr":
        payload = json.dumps({
            "alert_type": random.choice(["malware_detected", "ransomware_behavior", "credential_theft"]) if is_true_positive else random.choice(event_sources["edr"]["alert_types"]),
            "process_name": random.choice(["powershell.exe", "cmd.exe", "python.exe", "svchost.exe", "explorer.exe", "chrome.exe", "notepad.exe"]),
            "process_hash": ''.join(random.choices('0123456789abcdef', k=64)),
            "parent_process": random.choice(["explorer.exe", "services.exe", "winlogon.exe", "svchost.exe"]),
            "file_path": random.choice(["C:\\\\Temp\\\\", "C:\\\\Users\\\\Public\\\\", "C:\\\\Windows\\\\System32\\\\", "/tmp/", "/var/log/"]) + f"file_{random.randint(1,999)}.{'exe' if random.random() > 0.5 else 'dll'}",
            "mitre_tactic": random.choice(["Initial Access", "Execution", "Persistence", "Privilege Escalation", "Defense Evasion", "Credential Access", "Discovery", "Lateral Movement", "Collection", "Exfiltration"])
        })
    elif source_type == "auth":
        payload = json.dumps({
            "action": random.choice(["login_failure", "account_lockout", "mfa_failure"]) if is_true_positive else random.choice(event_sources["auth"]["actions"]),
            "username": f"user_{random.randint(1,500)}@{cust['customer_name'].split()[0].lower()}.com",
            "auth_method": random.choice(event_sources["auth"]["auth_methods"]),
            "source_ip": random.choice(malicious_ips) if is_true_positive else f"192.168.{random.randint(0,255)}.{random.randint(1,254)}",
            "geo_location": random.choice(["US", "UK", "DE", "CN", "RU", "BR", "IN", "JP", "AU", "NG"]),
            "user_agent": random.choice(["Chrome/120", "Firefox/119", "Edge/120", "Python-requests/2.31", "curl/8.4"])
        })
    elif source_type == "email":
        payload = json.dumps({
            "verdict": random.choice(["phishing", "malware_attachment", "bec_attempt"]) if is_true_positive else random.choice(event_sources["email"]["verdicts"]),
            "sender": f"{'attacker' if is_true_positive else random.choice(['john','jane','info','support','billing'])}@{random.choice(['evil','legit','company','partner'])}{random.randint(1,99)}.com",
            "recipient": f"user_{random.randint(1,500)}@{cust['customer_name'].split()[0].lower()}.com",
            "subject": random.choice(["Urgent: Account Verification", "Invoice #12345", "Meeting Tomorrow", "Password Reset Required", "Q3 Report", "Action Required"]),
            "has_attachment": random.random() > 0.6,
            "has_links": random.random() > 0.4
        })
    else:  # cloud_audit
        payload = json.dumps({
            "service": random.choice(event_sources["cloud_audit"]["services"]),
            "action": random.choice(["DisableLogging", "DeleteBucket", "CreateAccessKey"]) if is_true_positive else random.choice(event_sources["cloud_audit"]["actions"]),
            "principal": f"arn:aws:iam::{'root' if is_true_positive else f'user/admin_{random.randint(1,50)}'}",
            "resource": f"arn:aws:{random.choice(['s3','ec2','iam'])}:::resource-{random.randint(1,999)}",
            "region": random.choice(["us-east-1", "us-west-2", "eu-west-1", "ap-southeast-1"]),
            "is_error": random.random() > 0.8
        })
    
    events_data.append({
        "event_id": str(uuid.uuid4()),
        "event_timestamp": event_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "source_type": source_type,
        "customer_id": cust["customer_id"],
        "asset_id": asset["asset_id"],
        "severity_score": severity,
        "is_true_positive": is_true_positive,
        "raw_payload": payload
    })

# Land events as raw JSON files — one folder per day, like a SIEM/EDR export.
# These files are ingested into the bronze table by Auto Loader (02a_autoloader_ingest).
print(f"Creating DataFrame with {len(events_data)} events...")
events_df = spark.createDataFrame(events_data)
(events_df
    .withColumn("event_day", F.substring("event_timestamp", 1, 10))
    .repartition("event_day")
    .write.mode("overwrite").partitionBy("event_day")
    .json(LANDING_PATH))

landed = spark.read.json(LANDING_PATH)
count = landed.count()
num_files = len([f for d in dbutils.fs.ls(LANDING_PATH) if d.isDir() for f in dbutils.fs.ls(d.path) if f.name.endswith(".json")])
print(f"✅ Landed {count:,} raw security events as {num_files} JSON files in {LANDING_PATH}")

# Show distribution
landed.createOrReplaceTempView("landed_events")
spark.sql(f"""
    SELECT source_type,
           COUNT(*) as event_count,
           ROUND(COUNT(*) * 100.0 / {count}, 1) as pct,
           SUM(CASE WHEN is_true_positive THEN 1 ELSE 0 END) as true_positives
    FROM landed_events
    GROUP BY source_type
    ORDER BY event_count DESC
""").display()

Creating DataFrame with 500000 events...
✅ Landed 500,000 raw security events as 30 JSON files in /Volumes/rajarora_febar/soc_alert_triage/landing/security_events


source_type,event_count,pct,true_positives
firewall,175021,35.0,8902
auth,124891,25.0,6292
edr,99905,20.0,4894
email,50145,10.0,2485
cloud_audit,50038,10.0,2444


In [0]:
# These represent past analyst decisions on alerts.
# Analyst severity is driven by the alert's signals (plus noise), mimicking how a
# senior analyst weighs severity, asset criticality, threat intel and anomalies.
# NOTE: because labels are synthetic, the model learns this latent policy back.
import math

triage_outcomes = ["true_positive", "false_positive", "benign", "needs_investigation"]
triage_severities = ["critical", "high", "medium", "low", "informational"]
analyst_names = [f"analyst_{i}" for i in range(1, 21)]
outcome_notes = {
    "true_positive": ["Confirmed malicious - incident created", "Escalated to Tier 2"],
    "false_positive": ["Standard false positive pattern", "Duplicate alert - suppressed", "Automated scan triggered alert"],
    "benign": ["Known benign activity", "Customer-approved activity"],
    "needs_investigation": ["Requires additional context", "Escalated to Tier 2"],
}

def analyst_risk(f):
    """Latent analyst risk judgement from alert signals (higher = more severe)."""
    return (
        0.5 * f["original_severity"]
        + 0.5 * f["asset_criticality_score"]
        + 0.03 * f["user_risk_score"]
        + 2.5 * f["threat_intel_match"]
        + 1.2 * f["geo_anomaly"]
        + 0.8 * f["time_anomaly"]
        - 2.5 * f["historical_fp_rate"]
        - 0.08 * f["repeat_alert_count"]
        + random.gauss(0, 0.6)
    )

def severity_from_risk(r):
    if r >= 6.0: return "critical"
    if r >= 4.8: return "high"
    if r >= 3.6: return "medium"
    if r >= 2.4: return "low"
    return "informational"

triage_data = []
for i in range(50000):
    hour = random.randint(0, 23)
    features = {
        "original_severity": random.randint(1, 10),
        "hour_of_day": hour,
        "is_weekend": random.random() < 0.28,
        "repeat_alert_count": min(30, int(random.expovariate(1 / 1.5))),
        "user_risk_score": round(min(100.0, max(0.0, random.gauss(40, 15))), 1),
        "asset_criticality_score": random.choice([1, 2, 3, 4]),
        "historical_fp_rate": round(random.betavariate(6, 4), 3),
        "geo_anomaly": random.random() < 0.08,
        "time_anomaly": hour < 6,  # overnight activity
        "threat_intel_match": random.random() < 0.05,
    }
    risk = analyst_risk(features)
    is_tp = random.random() < 1 / (1 + math.exp(-1.5 * (risk - 6.4)))  # ~15% true positives
    outcome = "true_positive" if is_tp else random.choices(
        ["false_positive", "benign", "needs_investigation"],
        weights=[0.5 + features["historical_fp_rate"], 0.6, 0.2 + max(0, risk - 3.6) * 0.3],
    )[0]

    triage_data.append({
        "triage_id": f"TRIAGE-{i+1:06d}",
        "event_id": str(uuid.uuid4()),
        "customer_id": random.choice(customers_data)["customer_id"],
        "source_type": random.choice(list(event_sources.keys())),
        "analyst_severity": severity_from_risk(risk),
        "triage_outcome": outcome,
        "analyst_id": random.choice(analyst_names),
        "triage_time_minutes": round(random.uniform(15, 90), 1) if is_tp else round(random.uniform(2, 45), 1),
        "triage_timestamp": (base_time - timedelta(days=random.randint(1, 180))).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "notes": random.choice(outcome_notes[outcome]),
        **features,  # Features for ML training
    })

triage_df = spark.createDataFrame(triage_data)
triage_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLES["historical_triage"])
print(f"✅ Created {triage_df.count():,} historical triage decisions")

# Show outcome distribution
spark.sql(f"""
    SELECT triage_outcome,
           COUNT(*) as count,
           ROUND(AVG(triage_time_minutes), 1) as avg_triage_minutes
    FROM {TABLES['historical_triage']}
    GROUP BY triage_outcome
    ORDER BY count DESC
""").display()

# Sanity check: labels should track the signals (not be uniform)
spark.sql(f"""
    SELECT analyst_severity,
           COUNT(*) AS count,
           ROUND(AVG(original_severity), 2) AS avg_original_severity,
           ROUND(AVG(CAST(threat_intel_match AS INT)) * 100, 1) AS pct_threat_intel,
           ROUND(AVG(CASE WHEN triage_outcome = 'true_positive' THEN 1 ELSE 0 END) * 100, 1) AS pct_true_positive
    FROM {TABLES['historical_triage']}
    GROUP BY analyst_severity
    ORDER BY avg_original_severity DESC
""").display()

✅ Created 50,000 historical triage decisions


triage_outcome,count,avg_triage_minutes
false_positive,22701,23.4
benign,12256,23.5
needs_investigation,7579,23.4
true_positive,7464,52.5


analyst_severity,count,avg_original_severity,pct_threat_intel,pct_true_positive
critical,7779,8.51,18.8,65.3
high,9755,7.49,5.3,18.6
medium,11414,5.86,3.2,4.3
low,10445,4.06,1.2,0.7
informational,10607,2.4,0.2,0.0


## ✅ Data Generation Complete!

### Tables Created:
| Table | Records | Description |
|---|---|---|
| `customers` | 100 | MDR customer accounts |
| `threat_intel_iocs` | 10,000 | Known-bad indicators of compromise |
| `asset_inventory` | 5,000 | Managed endpoints and servers |
| `landing` volume | 500,000 events | Raw security telemetry as daily JSON files (30 days) |
| `historical_triage_decisions` | 50,000 | Past analyst triage decisions (for ML) |

**Next step**: Run `02a_autoloader_ingest` to ingest the landed files into bronze with Auto Loader.